# Final Report: Speech Emotion Recognition using the RAVDESS Dataset

**Author:** Brad Martin  
**Course:** CSE 432/532 — Machine Learning  
**Semester:** Spring 2026

---

This is my final report for the SER project. Over the semester I built a from scratch machine learning library called MiniLearn and used it together with SKlearn to classify emotions from audio files in the RAVDESS dataset. This notebook compiles all of my weekly work into one document with the analysis from each step, the comparisons between models, and my overall takeaways from the project. Most of the actual code is in the individual weekly notebooks and the minilearn folder, this one is more of a writeup that pulls everything together.

The goal of the project was to take audio files of people saying lines with different emotions and figure out which emotion they were expressing. The dataset has 8 emotions: neutral, calm, happy, sad, angry, fearful, disgust, and surprised. Each audio file is around 3-4 seconds long and there are around 2451 of them after I removed one corrupted file. I went through the entire pipeline starting from downloading the WAV files all the way to training neural networks on the extracted features.

## 1. Introduction

### What is Speech Emotion Recognition

Speech Emotion Recognition or SER is the task of taking an audio recording of someone talking and figuring out what emotion they are expressing. This is harder then it sounds because the same words can be said in tons of different ways and the emotion is mostly in the tone, energy, pitch, and rhythm rather then the words themselves. SER is used in things like call center analytics, mental health apps, voice assistants that try to detect if someone is frustrated, and a bunch of other places where understanding how someone is feeling matters more than just what they are saying.

### The RAVDESS dataset

The dataset I used is RAVDESS which stands for the Ryerson Audio-Visual Database of Emotional Speech and Song. It has 24 different actors (12 male and 12 female) each saying two fixed sentences ("Kids are talking by the door" and "Dogs are sitting by the door") in 8 different emotions and 2 different intensity levels. There is also a song version where the actors sing the lines. Actor 18 dosent have any of the song version which I had to handle in my code. The data was downloaded from Zenodo as two zip files, one for speech and one for song. Each file has a 7 part naming convention like 03-01-05-01-02-01-12.wav where each number means something different (modality, vocal channel, emotion, intensity, statement, repetition, actor).

### Project structure

My project is split into two main parts. The first is the minilearn folder which is my from scratch python package. It has all the classifiers, preprocessing tools, metrics, and other utilities I built throughout the semester. The second is the notebooks folder which has all the analysis notebooks where I actually use minilearn to do the SER classification and compare it against SKlearn. The general pattern every week was to implement something in minilearn, run it on the SER data, compare against the SKlearn version, and write up what I found.

## 2. Data Acquisition and EDA (Weeks 4-5)

### Loading the dataset

The first thing I did was setup the project structure and used the provided download_data.py script that grabs the two audio zip files from Zenodo and unzips them into a data folder. Then I wrote initialDataPipeline.py that goes through every WAV file, parses the 7 part filename, and builds a metadata CSV with one row per audio file. Each row has the modality, vocal channel, emotion, intensity, statement, repetition, actor, gender, and the file path.

Parsing the filenames was straightforward but I had to build maps for each number to its actual meaning (like 05 means angry) so the metadata would be readable. Gender is figured out by checking if the actor number is odd or even, odd means male and even means female.

### EDA findings

After building the metadata I made distribution plots for every column to see what the dataset actually looks like. A few things I found:

- **Emotions are not balanced.** Calm, happy, sad, angry, and fearful all have around 375 audio files each, but neutral, disgust, and surprised only have around 175 each. I knew this was going to make classification harder for those three because there is just less data to learn from. This came up over and over in my later analysis.
- **Speech vs song split.** There are around 1420 speech files and 1050 song files. So speech is more common which makes sense given actor 18 has no song files and song has 4 less files per actor.
- **Intensity is also imbalanced.** Around 1350 normal files and 1100 strong files. Neutral emotion only exists at normal intensity which makes sense, you cant have a strong neutral.
- **Gender is roughly even.** Within 50 samples between male and female.
- **Statement and repetition are basically 50/50.** No surprises there.

### Feature extraction (Week 5)

For feature extraction I used librosa to pull out 10 different feature types per audio file: MFCCs, MFCC deltas, MFCC delta-deltas, chroma, mel spectrogram, RMS, spectral centroid, bandwidth, ZCR, and rolloff. For each one I computed 4 summary statistics (mean, std, min, max) so each audio file ended up as a 480 dimensional feature vector. The final features csv has 2451 rows and 480 features.

The feature distribution plots showed some clear patterns. MFCC_1 mean had angry, fearful, and happy with the highest values and largest spread. ZCR mean had a curve peaking around angry and coming back down for surprised. RMS mean had angry, happy, and fearful with the highest energy which made physical sense because those are the louder emotions. Spectral centroid, rolloff, and chroma 1 had the opposite pattern, dipping around the loud emotions and being higher for surprised and neutral.

The correlation heatmap showed what I expected, the MFCCs are correlated with each other and the spectral features are correlated with each other. The off diagonal correlations between MFCC_1 mean and ZCR mean were interesting to me, I wouldnt have expected that going in. Overall there was a lot of feature redundancy which made me think dimensionality reduction would be useful later (which it was, see section 8).

## 3. Regression (Week 6)

For week 6 I used the built in SKLEARN linear regression and tried to predict the emotional intensity (1 = normal, 2 = strong) from the audio features. This was actually pretty hard because linear regression isnt really built for predicting between two integer values, its meant for continuous outputs.

I went through 3 rounds of iteration on this:

**Round 1:** Trained on all 480 features and got a test MSE of 5.19 and test R^2 of -20.03 which I didnt even think was possible. The negative R^2 means my model was worse then just predicting the mean every time. The problem was that with 480 features and only 2450 rows the model was massively overfitting.

**Round 2:** Took out the neutral emotion from the dataset because predicting intensity on neutral dosent make sense, all neutral files are normal intensity. Test MSE dropped to 4.15 and R^2 was -15.6. Still bad.

**Round 3:** Removed every feature that didnt end in _mean. So no std, min, or max features just the means. This dramatically cut down the feature count and helped a ton with the overfitting. Test MSE was 0.19 (slightly higher then train at 0.17) and R^2 was 0.30 on train and 0.24 on test. These numbers arent great but they actually make sense.

The takeaway here was that linear regression really isnt the right tool for a basically binary classification problem like normal vs strong intensity. Logistic regression would have been a better fit and I built that in the next week. But it was a useful exercise in seeing how overfitting works with too many features and how feature selection helps.

## 4. Classification: LR, KNN, NB (Week 7)

This was the first week where I really started building the supervised classifiers in minilearn and comparing them against SKlearn. I built logistic regression with softmax (for multiclass), KNN, and Gaussian Naive Bayes all from scratch.

### Logistic Regression

My minilearn LR uses softmax for multiclass classification and gradient descent for optimization. When I first ran it the accuracy was higher then SKlearn which seemed suspicious. After digging into it I figured out that my model was running fewer iterations then SKlearn so it was less overfit. When I cranked both up to 5000 iterations my model just barely beats SKlearn but the difference is small enough that I think its essentially the same. I think the small edge comes from my model being slightly less complex and not overfitting as hard.

The easiest emotions to classify were angry, calm, and happy. The hardest were disgust, neutral, and surprised. This pattern shows up in every single model I built which makes sense because those three emotions also have the fewest samples in the dataset.

### KNN

My minilearn KNN matched SKlearn closely because KNN is simple, its just measuring distance between points and voting on the nearest neighbors. Both models got around 58% accuracy which isnt great. From what I read KNN struggles with high dimensionality which is exactly my situation, 480 features and only 2400 samples. The curse of dimensionality means distances between points all start to look the same when you have too many features which breaks KNN. Same emotion pattern showed up, angry and calm were easiest, neutral disgust and surprised were hardest.

### Gaussian Naive Bayes

This was the hardest model for me to understand. NB assumes the features are independent (no correlation) and follow a normal distribution. Both of these assumptions are wildly wrong for my data, my correlation heatmap showed tons of correlated features and the per-emotion distributions are not normal. My minilearn version and SKlearn both got really bad accuracy around 0.275 which was the worst of any model I built. The confusion matrix showed it was predicting basically everything as angry, disgust, or happy which is a big sign the model isnt actually learning the patterns.

The takeaway from this week was that LR is a strong baseline that competes with SKlearn, KNN is simple and works okay but struggles with high dim data, and NB is a bad fit for correlated audio features.

## 5. SVM (Week 8)

For week 8 I built a simplified linear SVM in minilearn. The assignment said simplified linear so I made mine binary, it uses hinge loss with gradient descent and maps the two classes to -1 and +1. For the binary test I picked happy vs sad as the two emotions and my model competed pretty well with SKlearns LinearSVC. My train accuracy was lower then SKlearns but my test accuracy was also lower by a small amount. SKlearns 100% train accuracy made me think it was overfitting which I think made my model the slightly more generalized version.

For the multiclass full 8 emotion comparison I used SKlearns SVC with three different kernels: linear, RBF, and polynomial. RBF was the clear winner.

- **Linear:** test accuracy 0.6980, macro F1 0.7000, but train accuracy was 1.0 which is the same overfitting issue I saw with my binary model
- **RBF:** test accuracy 0.7224, macro F1 0.7230, the highest scores out of any model
- **Poly:** test accuracy 0.5776, macro F1 0.5409, the worst of the three

RBF won because it can model non-linear patterns in the data without overfitting as badly as the linear model. Emotion classes arent linearly separable, you cant draw a straight line between angry and happy because they share a lot of features (both high energy). RBF projects the data into an infinite dimensional space where curved decision boundaries become possible which is exactly what this task needs.

I also ran a grid search on the RBF model to tune hyperparameters. It found C=10 and gamma=scale as the best combo which boosted macro F1 from 0.7230 to 0.7711 which is a 5% improvement. So tuning actually made a real difference here unlike with LR where it barely moved the needle.

## 6. Decision Trees and Ensembles (Week 9)

For week 9 I built CART (Classification And Regression Trees) from scratch in minilearn. CART works by recursively finding the best feature and threshold to split the data on, using gini impurity as the cost function. The tree keeps splitting until it hits a max depth or theres not enough samples to split.

### Single CART tree

My CART was painfully slow on the full 480 feature dataset. It was taking 45+ minutes per tree because for each split it has to check every feature and every possible threshold. So I ran it on a 20% subset of the data for the demonstration. Even on the subset my model got worse accuracy then SKlearns Decision Tree because I had to use max_depth=5 to keep it tractable while SKlearn could go deeper since its written in C and runs way faster.

The confusion matrix from a single CART tree looked like someone threw darts at a dartboard. It was barely better then random for most emotions. The fundamental problem with single trees is that the first couple splits dominate everything else. If those early splits arent good then the whole rest of the tree is messed up. Single trees also overfit really easily, you can see this in the gap between train and test accuracy.

### Random Forest

Random forest fixes all of the single tree problems. Instead of one tree it builds a ton of them (I used 1000) and each tree only sees a random subset of features and a random subset of samples. Then it majority votes across all trees for the final prediction. The bad trees get outvoted by the good ones and the noise averages out.

Random forest was way better then single CART. Train accuracy was super high which would normally be an overfitting concern but the test accuracy was also solid so it was actually learning patterns rather then memorizing. The confusion matrix had a clean diagonal showing it was correctly classifying most emotions. Same pattern as always though, angry/calm/happy easy and neutral/disgust/surprised hard.

### AdaBoost

AdaBoost is a different ensemble approach. Instead of training a bunch of independent trees and voting, AdaBoost trains them one at a time with each new tree focusing on the samples the previous ones got wrong. The individual trees are very shallow (I used depth 1 which means basically stumps with one split each).

AdaBoost did okay but not as good as Random Forest. From what I read this is common with multiclass problems, AdaBoost was originally built for binary and the multiclass version dosent always work as well. It also overfits to noisy data because it keeps amplifying the weight of hard examples and if some of those are actually just outliers it ends up tuning to them.

The ranking from worst to best was: single CART → AdaBoost → Random Forest. The big takeaway is that combining many weak learners gets you somewhere much better then any single weak learner.

## 7. Model Validation (Week 10)

Up to this point every model was evaluated on a single 80/20 train test split. Thats okay for getting a rough number but it dosent tell you how stable that number is. K-fold cross validation fixes this by running each model on k different splits and averaging the results.

I built my own k_fold_split in minilearn that does stratified splitting (each fold has the same class balance as the full dataset). For the actual validation notebook I built a helper run_kfold that takes a model factory and runs it through 5-fold CV with the scaler fitted inside the loop on each training fold separately.

I ran 5-fold CV on every classifier with default parameters and then built my own grid search on top of k-fold to find better hyperparameters. The grid search loops through every combination, scores each one by mean macro F1 across folds, and returns the best.

### Key findings from the validation notebook

- **Per fold std was small for most models** (around 0.01 to 0.03) which means my single split numbers from the earlier notebooks were already pretty reliable. K-fold confirmed those numbers werent flukes.
- **Tuning helped but not by a lot.** For LR going from default to tuned only moved macro F1 by 1-2 points. KNN was the same. This tells me the features are the bottleneck not the hyperparameters. To meaningfully improve the models I would need better features (like pretrained embeddings) not better tuning.
- **CART had the biggest spread across folds** which makes sense because trees are notoriously unstable, small changes in data create completely different trees.
- **The class imbalance issue continues to show up.** Macro F1 was lower then accuracy for every model because of the small classes (neutral, disgust, surprised) dragging the macro average down.

### Notes on the validation setup

- CART was run on a 20% subset because of how slow it was on full data. The numbers are slightly lower then they would be on full data but it gives me a relative comparison. 

## 8. Clustering (Week 11)

This week was the unsupervised part of the project. The goal was to see if K-Means could recover the 8 emotion classes without being told what the labels are. Spoiler: it really cant, and the reason why is one of the most interesting findings in the whole project.

### K-Means implementation

I built K-Means from scratch in minilearn. It works by picking k random points as starting centroids, assigning every point to its nearest centroid, recomputing the centroids as the mean of their assigned points, and repeating until things stop changing. I compared mine against SKlearns KMeans on the same data.

The cluster size distributions were different between the two. My version had clusters ranging from 122 to 526 in size while SKlearn had clusters from 1 to 512. SKlearn had a cluster of size 1 which is a degenerate cluster where one outlier was alone, this happens because SKlearn uses n_init=10 (best of 10 random starts) and minimizes inertia which can produce ugly cluster sizes. My version uses a single random init so it tends to be more balanced but maybe not as optimal.

### ARI and NMI

To measure how well the clusters matched the actual emotions I used Adjusted Rand Index and Normalized Mutual Information. Both were low (around 0.05-0.15 for ARI and 0.10-0.20 for NMI) which is what the literature says you should expect when clustering audio features by emotion. This confirmed that the clusters did not align with emotion.

### The big finding: gender dominates the features

The most important finding from this entire project came from the t-SNE plot. When I colored the t-SNE projection by cluster, K-Means showed clear distinct neighborhoods. When I colored by true emotion, it was a uniform mess of all 8 colors mixed together. But when I colored by **gender**, the male and female samples split almost perfectly down the middle of the plot.

What this is telling me is that the biggest source of variance in my features is not emotion at all, its just whos talking. Male and female voices differ acoustically (males around 100hz, females double that around 200hz) and a lot of my features (especially MFCCs and spectral stuff) are measuring the shape of the vocal tract which is heavily tied to gender. So gender is baked into the features before emotion gets a chance to show up.

K-Means optimizes Euclidean distance which makes it biased toward the biggest source of variance. It locks onto gender and never reaches emotion. This explains why my ARI/NMI were so low, theres a much bigger signal in the way of the emotion signal.

This is actually consistent with the speech emotion recognition literature. Speaker-independent emotion classification fundamentally requires speaker normalization, gender conditional modeling, or pretrained representations like wav2vec or HuBERT that are specifically trained to be speaker invariant. Hand-crafted MFCC-style features cant escape speaker dominance in an unsupervised setting.

## 9. Dimensionality Reduction with PCA (Week 12)

I have 480 features and a lot of them are correlated (MFCC mean and MFCC std move together, spectral features all track loudness, etc). So this was a good situation for PCA.

### PCA implementation

My minilearn PCA does the standard eigendecomposition of the covariance matrix. First it centers the data, computes the covariance matrix, finds the eigenvectors and eigenvalues, sorts them by variance explained, and takes the top k components. I checked that my version matches SKlearn by comparing the explained variance ratios.

### Explained variance

The variance drops off pretty fast after the first few components. The thresholds I cared about:
- 80% variance: around 35-40 components
- 90% variance: around 60-70 components  
- 95% variance: around 100-120 components
- 99% variance: around 250 components

So I could potentially cut my features from 480 down to 70 and still capture 90% of the variance which is a huge reduction.

### 2D visualization

When I projected the data onto the first 2 components and colored by emotion, I got the same mess I saw in the clustering notebook. The first 2 components only capture about 30-35% of variance so emotion seperation in 2D was never going to look clean. PC1 is probably picking up gender or loudness based on the clustering analysis, and PC2 is probably some spectral shape thing.

### Re-running classifiers on PCA reduced features

This is the part the readme specifically asked for. I ran a sweep of component counts (2, 10, 25, 50, 100, 150, 200) and trained LR, KNN, NB, SVM, and CART on each. My hypothesis was:
- At 2 or 10 components everything would be bad because too much info is gone
- Around 50-100 components performance would match or beat the full features because the noise correlations are gone
- Past 100 returns would diminish

The hypothesis kind of held up. KNN benefited the most from PCA which makes sense, KNN suffers in high dimensions because distances all start to look the same so reducing dimensions actually helps it. CART got slightly worse with PCA which also makes sense, trees like to split on individual informative features and PCA components are linear mixes that wash out those single feature signals. LR and NB stayed about the same.

### Headline finding

For most models PCA matches the full feature performance but dosent really beat it. My features arent so redundant that PCA finds free accuracy. But the training time drops a lot which is the actual practical benefit. With 70 features instead of 480 the models train faster and would scale better.

The real takeaway is that the bottleneck for my models isnt the dimensionality, its the quality of the features themselves. PCA is a fast cheap version of the same information.

# Neural Network

### Time

To be honest with this assignment I am already turning it in late because I started the project to late. I have probably close to 70 hours in the last week and half into the project and I just didnt have time spending another 10-12 hours on the nueral network component of the project. I apologize

## 11. Model Comparison Summary

Pulling together the macro F1 scores across all models tested in this project:

| Model | Type | Macro F1 (approx) | Notes |
|---|---|---|---|
| RBF SVM (tuned) | SKlearn | 0.77 | Best overall, C=10, gamma=scale |
| RBF SVM (default) | SKlearn | 0.72 | Strong baseline |
| Random Forest | SKlearn | 0.65 | 1000 trees, depth 20 |
| Linear SVM | SKlearn | 0.70 | Overfit training set |
| Logistic Regression | MiniLearn | 0.55 | Beat SKlearn slightly when iters=5000 |
| Logistic Regression | SKlearn | 0.54 | Standard implementation |
| AdaBoost | SKlearn | 0.50 | Stump base learners |
| KNN k=5 | MiniLearn | 0.58 | Hurt by high dim features |
| KNN k=5 | SKlearn | 0.58 | Matched minilearn closely |
| Polynomial SVM | SKlearn | 0.54 | Worst SVM kernel |
| Single CART tree | MiniLearn | 0.35 | Slow, unstable |
| Gaussian NB | MiniLearn | 0.28 | Hurt by correlated features |

### Best model: RBF SVM

The RBF kernel SVM with tuned C=10 was the clear winner. It can model non-linear decision boundaries which is exactly what emotion classification needs. Linear models couldnt separate angry from happy because they share too many features (both high energy, both high pitch). RBF projects into a curved decision space where those overlaps can be resolved.

### Per-emotion performance

The same pattern showed up in every single model:
- **Easy emotions:** angry, calm, happy — these have distinct acoustic signatures (loudness, energy, pitch contour)
- **Hard emotions:** neutral, disgust, surprised — these have fewer training samples (175 vs 375) and acoustically overlap with the easier ones

The class imbalance is the dominant factor for the hard emotions. With half as many training samples the models just dont have enough data to learn what makes those emotions distinct.

### MiniLearn vs SKlearn

For most models my minilearn implementations were within a few percentage points of SKlearn. The places where SKlearn won were:
- **CART** because SKlearn uses optimized C code, my python implementation was 100x slower
- **RBF/Poly SVM** because the RBF kernel needs a quadratic programming solver which is way more complex then what I implemented
- **Random Forest** which I didnt implement in minilearn at all per the assignment

The places where my minilearn essentially tied or barely beat SKlearn:
- **Logistic Regression** with enough iterations
- **KNN** because the algorithm is simple, nothing to optimize
- **PCA** because its just eigendecomposition
- **Gaussian NB** both implementations were similarly bad on this data